# MLP: Modelado de Fronteras de Decisión No Lineales

**Comparación experimental entre un modelo lineal y una red neuronal multicapa**

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luisf2020/Proyectos-de-Inteligencia-Artificial/blob/main/notebooks/06_fronteras_no_lineales_mlp_iris.ipynb)

> Implementación desarrollada específicamente para este repositorio como proyecto técnico de portafolio.

## Objetivo

Comparar una frontera lineal con una frontera aprendida por un Perceptrón Multicapa (MLP). Usaremos dos características de Iris para poder representar el espacio de decisión en dos dimensiones.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [ ]:
iris = load_iris(as_frame=True)
features = ["petal length (cm)", "petal width (cm)"]
X = iris.data[features]
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", Perceptron(max_iter=2500, random_state=42))
])

mlp_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(24, 12),
        activation="tanh",
        solver="lbfgs",
        max_iter=3000,
        random_state=42
    ))
])

linear_model.fit(X_train, y_train)
mlp_model.fit(X_train, y_train)

for name, model in [("Perceptrón", linear_model), ("MLP", mlp_model)]:
    prediction = model.predict(X_test)
    print(f"{name:12s} accuracy: {accuracy_score(y_test, prediction):.3f}")

In [ ]:
def decision_surface(ax, model, X, y, title):
    x_min, x_max = X.iloc[:, 0].min() - 0.5, X.iloc[:, 0].max() + 0.5
    y_min, y_max = X.iloc[:, 1].min() - 0.3, X.iloc[:, 1].max() + 0.3

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 350),
        np.linspace(y_min, y_max, 350)
    )
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=X.columns)
    zz = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, zz, alpha=0.25)
    for label, name in enumerate(iris.target_names):
        mask = y == label
        ax.scatter(
            X.loc[mask, X.columns[0]],
            X.loc[mask, X.columns[1]],
            label=name,
            edgecolor="black",
            s=45
        )
    ax.set_xlabel(X.columns[0])
    ax.set_ylabel(X.columns[1])
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
decision_surface(axes[0], linear_model, X, y, "Perceptrón lineal")
decision_surface(axes[1], mlp_model, X, y, "Perceptrón Multicapa")
axes[1].legend()
plt.tight_layout()
plt.show()

## Conclusiones

La comparación permite observar el beneficio principal de incorporar capas ocultas: el MLP puede modelar relaciones no lineales y formar regiones de decisión más flexibles. Esa capacidad también aumenta la complejidad del modelo, por lo que la validación y el control del sobreajuste cobran mayor importancia.